# Predicted Epitopes of Trypanosoma cruzi Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2 dataset](https://doi.org/10.71728/senscience.etbd-dths) using the `mlcroissant` library, referencing dataset entities strictly by their `@id` values for reproducibility.

### Dataset Source
The dataset source Croissant schema is provided via URL and is machine-actionable.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.etbd-dths/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as object attributes
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Published: {meta.datePublished}")
print(f"License: {meta.license}")

## 2. Data Overview
Review available record sets, their IDs, the field `@id`s, and summary information. All references are made using the `@id` field of each entity.

In [ ]:
# List available record sets by their @id, with field (column) @ids
print("Available Record Sets (by @id) and their Fields:")

for rs in dataset.record_sets:
    print(f"- Record Set @id: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Field @ids:")
        for field in rs.fields:
            print(f"    - {field.id} (name: {field.name})")
    else:
        print("  No fields recorded.")
    print("")

## 3. Data Extraction
Load data from selected record sets into Pandas DataFrames for analysis.

Below, we collect all record set `@id`s, and demonstrate loading data from the first one for further exploration.

In [ ]:
# Gather all record set @ids
record_sets = [rs.id for rs in dataset.record_sets]
print(f"Record set @ids: {record_sets}\n")

# Load records from each record set into dataframes and report columns by @id
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records) if records else pd.DataFrame()
    dataframes[record_set_id] = df
    print(f"Record set @id '{record_set_id}' columns: {df.columns.tolist()}")

# Choose the first record set (customize this @id for analysis below)
selected_rs = record_sets[0] if record_sets else None

if selected_rs and not dataframes[selected_rs].empty:
    print(f"\nPreview of records from record set @id: {selected_rs}")
    display(dataframes[selected_rs].head())
else:
    print("No data records found in provided record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply basic EDA steps: filtering, normalization, and grouping using field `@id` values. Edit the variable assignments below to point to relevant `@id`s for numeric and grouping fields.

In [ ]:
import numpy as np

# Choose which record set and numeric field to analyze (edit as needed based on above printout)
record_set_id = selected_rs
df = dataframes.get(record_set_id, pd.DataFrame())

# Example: Use a field @id known to be numeric, customize as needed
example_numeric_field_id = None
example_group_field_id = None
if not df.empty:
    # Attempt to pick a likely numeric field by inspecting datatypes
    for col in df.columns:
        # If the column can be converted to numeric, select it
        try:
            df[col] = pd.to_numeric(df[col])
            if np.issubdtype(df[col].dtype, np.number):
                example_numeric_field_id = col
                break
        except:
            continue

    # Attempt to pick a group-by categorical field
    for col in df.columns:
        if df[col].dtype == object and df[col].nunique() < df.shape[0]/2:
            example_group_field_id = col
            break

if example_numeric_field_id:
    print(f"Selected numeric field @id: {example_numeric_field_id}")
    # Filter, normalize, group
    threshold = np.nanmean(df[example_numeric_field_id])
    filtered_df = df[df[example_numeric_field_id] > threshold]
    print(f"Filtered records with {example_numeric_field_id} > {threshold:.3f}")
    display(filtered_df.head())

    # Normalize numeric column
    col_norm = f"{example_numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[example_numeric_field_id] - filtered_df[example_numeric_field_id].mean()) / filtered_df[example_numeric_field_id].std()
    print(f"\nExample normalization of {example_numeric_field_id} field:")
    display(filtered_df[[example_numeric_field_id, col_norm]].head())

    # Group by a categorical field if present
    if example_group_field_id:
        grouped_df = filtered_df.groupby(example_group_field_id).mean(numeric_only=True)
        print(f"\nGrouped normalized data by field @id: {example_group_field_id}")
        display(grouped_df.head())
    else:
        print("No suitable categorical/group field found for grouping.")
else:
    print("No numeric field found in selected record set.")

## 5. Visualization
Visualize distributions or relationships using field `@id`s. Adjust the field ids as appropriate for your exploration.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_numeric_field_id and not df.empty:
    plt.figure(figsize=(8,5))
    sns.histplot(df[example_numeric_field_id], kde=True)
    plt.title(f"Distribution of field: {example_numeric_field_id}")
    plt.xlabel(example_numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If a grouping field exists, show boxplot
    if example_group_field_id:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[example_group_field_id], y=df[example_numeric_field_id])
        plt.xlabel(example_group_field_id)
        plt.ylabel(example_numeric_field_id)
        plt.title(f"{example_numeric_field_id} by {example_group_field_id}")
        plt.xticks(rotation=60)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, searching for record sets, basic filtering, normalization, and simple data visualization for the FAIR^2 Trypanosoma cruzi epitope dataset using the `mlcroissant` library. All data entities are referenced by their Croissant `@id` fields as best practice.

You can customize the selected record set or field `@id`s for deeper domain analysis or further ML-ready data engineering.